# Service Pattern Fix: Direction Inference & GTFS Mapping

This notebook explores two strategies to fix service inflation caused by missing `direction_id` in the current GTFS feed:
1.  **Dominant Headsign Analysis**: Identifying the top 2 headsigns per route to create a cleaner 2-direction split.
2.  **Cross-GTFS Mapping**: Attempting to map `direction_id` from a newer (2026) GTFS feed to the current feed.

We will also analyze if matching on `route_id` and `trip_headsign` is sufficient, or if route IDs have changed (requiring `route_short_name` matching).

In [1]:
# 1. Load Libraries & Data
import pandas as pd
import numpy as np
import zipfile
from pathlib import Path

# Paths
current_gtfs_path = "../data/external/study_area_gtfs_bus.zip"
# Update this path to where your new GTFS is located
new_gtfs_path = "../data/external/itm_yorkshire_gtfs.zip"

print(f"Current GTFS: {current_gtfs_path}")
print(f"New GTFS: {new_gtfs_path}")

Current GTFS: ../data/external/study_area_gtfs_bus.zip
New GTFS: ../data/external/itm_yorkshire_gtfs.zip


## 2. Load Current GTFS Data
We extract `trips.txt` and `routes.txt` to analyze the current state of headsigns and route IDs.

In [2]:
def load_gtfs_table(zip_path, table_name):
    with zipfile.ZipFile(zip_path) as z:
        if f"{table_name}.txt" in z.namelist():
            return pd.read_csv(z.open(f"{table_name}.txt"))
        else:
            print(f"Warning: {table_name}.txt not found in {zip_path}")
            return None


# Load Current
df_trips_current = load_gtfs_table(current_gtfs_path, "trips")
df_routes_current = load_gtfs_table(current_gtfs_path, "routes")

print(f"Current Feed: {len(df_routes_current)} routes, {len(df_trips_current)} trips")
df_trips_current.head()

Current Feed: 187 routes, 13974 trips


,route_id,service_id,trip_id,trip_headsign,block_id,shape_id,wheelchair_accessible,trip_direction_name,vehicle_journey_code
0,50627,1221,VJ61453c073b835840fdf78144f238befa6c687ca8,Leeds Bradford Airport,NaN,RPSP328891cbcd5101c74194d673d7a0f4ecd66cd872,0,NaN,"VJ_0,VJ_22,VJ_43"
1,50627,1221,VJ4d4247975a1053cdd62e1903d53de7bde97d5c7c,Leeds Bradford Airport,NaN,RPSP328891cbcd5101c74194d673d7a0f4ecd66cd872,0,NaN,"VJ_1,VJ_23,VJ_44"
2,50627,1302,VJ5148f3c3e3804daef016cd5a7781405d6180e299,Leeds Bradford Airport,NaN,RPSP328891cbcd5101c74194d673d7a0f4ecd66cd872,0,NaN,"VJ_100,VJ_118"
3,50627,1302,VJ941e394228edbbbac82c32a74d5ea3d81e5091b7,Leeds Bradford Airport,NaN,RPSP328891cbcd5101c74194d673d7a0f4ecd66cd872,0,NaN,"VJ_101,VJ_119"
4,50627,1303,VJc57b708ca195f01b8682545b8f4dfa15eb5a7f21,Leeds Bradford Airport,NaN,RPSP328891cbcd5101c74194d673d7a0f4ecd66cd872,0,NaN,VJ_102


## 3. Load New GTFS Data (Reference)
Load the new GTFS feed to check if we can use its `direction_id` column.

In [3]:
# Load New
if Path(new_gtfs_path).exists():
    df_trips_new = load_gtfs_table(new_gtfs_path, "trips")
    df_routes_new = load_gtfs_table(new_gtfs_path, "routes")
    print(f"New Feed: {len(df_routes_new)} routes, {len(df_trips_new)} trips")
else:
    print("New GTFS file not found. Please ensure the path is correct.")
    df_trips_new = pd.DataFrame()  # Empty for safety
    df_routes_new = pd.DataFrame()

New Feed: 1180 routes, 169577 trips


## 4. Analyze Headsigns in Current GTFS
Identify routes with high headsign fragmentation (e.g., > 2 unique headsigns). These are the candidates for "Service Inflation" because missing direction_ids force the logical to treat every headsign as a new direction.

In [4]:
# Analyze current headsigns
headsign_counts = df_trips_current.groupby("route_id")["trip_headsign"].nunique()

print(f"Max headsigns for a single route: {headsign_counts.max()}")
print(f"Average headsigns: {headsign_counts.mean():.2f}")

# Distribution
print("\nNumber of routes with N headsigns:")
print(headsign_counts.value_counts().sort_index())

# Examples of fragmented routes
fragmented_routes = headsign_counts[headsign_counts > 2].index
print(f"\nRoutes with > 2 headsigns: {len(fragmented_routes)}")
if len(fragmented_routes) > 0:
    sample_route = fragmented_routes[0]
    print(f"Sample Route {sample_route} headsigns:")
    print(df_trips_current[df_trips_current.route_id == sample_route]["trip_headsign"].unique())

Max headsigns for a single route: 8
Average headsigns: 2.58

Number of routes with N headsigns:
trip_headsign
1     13
2    102
3     40
4     23
5      4
6      3
7      1
8      1
Name: count, dtype: int64

Routes with > 2 headsigns: 72
Sample Route 124 headsigns:
['Bradford City Centre' 'Victoria' 'Leeds City Centre' 'Hull']


## 5. Identify Dominant Headsigns
Strategy A: Can we fix the issue by just keeping the top 2 most frequent headsigns per route?
This section calculates how many trips would be covered if we only took the top 2 headsigns for each route.

In [5]:
def analyze_dominant_headsigns(df_trips):
    records = []

    for route_id, group in df_trips.groupby("route_id"):
        total_trips = len(group)
        headsign_counts = group["trip_headsign"].value_counts()

        # Take top 2
        top_2_count = headsign_counts.iloc[:2].sum()
        coverage = top_2_count / total_trips

        # Determine 3rd+ headsigns (the "noise")
        noise_trips = total_trips - top_2_count

        records.append(
            {
                "route_id": route_id,
                "total_trips": total_trips,
                "unique_headsigns": len(headsign_counts),
                "top_2_coverage": coverage,
                "noise_trips": noise_trips,
            }
        )

    return pd.DataFrame(records)


df_dominance = analyze_dominant_headsigns(df_trips_current)

print(f"Average coverage of top 2 headsigns: {df_dominance.top_2_coverage.mean():.2%}")
print(f"Routes with perfectly 2 or fewer headsigns: {len(df_dominance[df_dominance.unique_headsigns <= 2])}")
print(f"Routes where top 2 cover < 90% of trips: {len(df_dominance[df_dominance.top_2_coverage < 0.9])}")

# Show worst offenders
print("\nRoutes with worst Top-2 Coverage:")
print(df_dominance.sort_values("top_2_coverage").head())

Average coverage of top 2 headsigns: 95.50%
Routes with perfectly 2 or fewer headsigns: 115
Routes where top 2 cover < 90% of trips: 31

Routes with worst Top-2 Coverage:
     route_id  total_trips  unique_headsigns  top_2_coverage  noise_trips
120     33496           24                 8        0.416667           14
63      16835           15                 6        0.533333            7
185     77162           13                 6        0.538462            6
158     64692           18                 5        0.722222            5
67      19840           63                 7        0.730159           17


## 6. Attempt Mapping Direction ID from New GTFS
Strategy B: Map `direction_id` from the new GTFS.
We prioritize matching on `route_id` + `trip_headsign`. If `route_id` changed, we fallback to matching on `route_short_name` + `trip_headsign`.

In [ ]:
if not df_trips_new.empty:
    # 1. Prepare Reference Table from New Feed
    # Key: (route_id, trip_headsign) -> direction_id
    ref_directions_id = df_trips_new[["route_id", "trip_headsign", "direction_id"]].drop_duplicates()

    # 2. Attempt Direct Match on Route ID
    print("Attempting match on Route ID...")
    mapped_trips_id = pd.merge(
        df_trips_current, ref_directions_id, on=["route_id", "trip_headsign"], how="left", suffixes=("", "_new")
    )

    match_rate_id = mapped_trips_id["direction_id_new"].notna().mean()
    print(f"Match Rate (Route ID): {match_rate_id:.1%}")

    final_mapped = mapped_trips_id

    # 3. If match rate is low (< 50%), try Route Short Name fallback
    if match_rate_id < 0.5:
        print("\nMatch rate low. Attempting fallback match on Route Short Name...")

        # Prepare Route Map (Route ID -> Short Name) for both feeds
        routes_old = df_routes_current[["route_id", "route_short_name"]].rename(columns={"route_id": "route_id_old"})
        routes_new = df_routes_new[["route_id", "route_short_name"]].rename(columns={"route_id": "route_id_new"})

        # Prepare Reference Table using Short Name
        # Join new trips with new routes to get short name
        trips_new_named = pd.merge(df_trips_new, routes_new, left_on="route_id", right_on="route_id_new")
        ref_directions_name = trips_new_named[["route_short_name", "trip_headsign", "direction_id"]].drop_duplicates()

        # Check for conflicts in short name mapping (e.g. same short name used by different agencies/routes?)
        conflicts = ref_directions_name.groupby(["route_short_name", "trip_headsign"]).size()
        if (conflicts > 1).any():
            print(f"Warning: {sum(conflicts > 1)} headsign conflicts found in short name mapping.")

        # Clean conflicts
        ref_directions_name = ref_directions_name.drop_duplicates(subset=["route_short_name", "trip_headsign"])

        # Prepare Current Trips
        trips_current_named = pd.merge(df_trips_current, routes_old, left_on="route_id", right_on="route_id_old")

        # Match
        mapped_trips_name = pd.merge(
            trips_current_named,
            ref_directions_name,
            on=["route_short_name", "trip_headsign"],
            how="left",
            suffixes=("", "_new"),
        )

        match_rate_name = mapped_trips_name["direction_id_new"].notna().mean()
        print(f"Match Rate (Short Name): {match_rate_name:.1%}")

        if match_rate_name > match_rate_id:
            print("Using Short Name mapping results.")
            final_mapped = mapped_trips_name

    # 4. Final Analysis of Validation
    matched_count = final_mapped["direction_id_new"].notna().sum()
    total_count = len(final_mapped)
    print(f"\nFinal Mapping Success: {matched_count}/{total_count} ({matched_count / total_count:.1%})")

    # Check unmatched
    unmatched = final_mapped[final_mapped["direction_id_new"].isna()]
    if not unmatched.empty:
        print("\nTop unmatched headsigns:")
        print(unmatched["trip_headsign"].value_counts().head())

else:
    print("Skipping mapping section as new GTFS is not loaded.")

## 7. Explore Frequency Splitting Strategy
If we can't reliably map direction_ids, Strategy C is to split the frequency.
If a route has 4 "directions" (headsigns), we should run each at 1/4th the frequency (4x the headway) so the *aggregate* service matches the optimized headway.

The calculation: `Adjusted_Headway = Base_Headway * N_Directions`
Where `N_Directions` is the number of unique headsigns actively served in that time window.

In [ ]:
# Simulation: Effect of Headway Splitting
# Assume optimized headway is 10 mins.
base_headway = 10.0

df_sim = df_dominance.copy()
df_sim["active_branches"] = df_sim["unique_headsigns"]  # Assuming all run all day for worst case

# Current behavior: 10 mins on EACH branch
# Aggregate Frequency = Sum(1/10) * N = N/10 trips per minute
# Aggregate Headway = 10 / N
df_sim["current_agg_headway"] = base_headway / df_sim["active_branches"]

# Proposed fix: Split frequency
# We want Aggregate Headway = 10.
# So each branch needs Headway = 10 * N
df_sim["required_branch_headway"] = base_headway * df_sim["active_branches"]

print("Simulation Results (Target Aggregate Headway = 10 mins):")
print(df_sim[["route_id", "unique_headsigns", "current_agg_headway", "required_branch_headway"]].head(10))

print("\nImpact Summary:")
print(f"Routes with < 5 min aggregate headway currently: {len(df_sim[df_sim.current_agg_headway < 5])}")
print(f"Max Inflation Factor: {df_sim.unique_headsigns.max()}x service")

## 8. Conclusion and Next Steps

Based on the analysis above:
1. **Dominant Headsigns:** If >90% of trips are covered by the top 2 headsigns, we can force a 2-direction split in `gtfs.py` by filtering out the "noise" headsigns.
2. **GTFS Mapping:** If the `route_short_name` based mapping works for a high percentage of trips, we can inject the `direction_id` from the 2026 feed into our current data.
3. **Frequency Splitting:** If all else fails, the frequency splitting method provides a mathematically sound way to ensure the *aggregate* service level is correct, even if the "directions" are fragmented.

**Action Plan:**
- If mapping works well -> Implement mapping script to enrich current GTFS.
- If mapping fails but dominant headsigns are clear -> Update `gtfs.py` to filter for top-2 headsigns per route.
- If neither works well -> Update `gtfs.py` to split `headway` by `N_directions`.